# Clinic task decomposition: starter

Loads the per-form results and compares field accuracy across the number of sub-calls.

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

matches = glob.glob('/kaggle/input/**/per_run.csv', recursive=True) or glob.glob('**/per_run.csv', recursive=True)
assert matches, 'per_run.csv not found; attach the dataset first'
runs = pd.read_csv(matches[0])
print(os.path.dirname(matches[0]), runs.shape)
runs.head()

In [ ]:
summary = (runs.groupby(['model', 'language', 'dependency', 'granularity'])
           .agg(items=('item_id', 'nunique'), field_accuracy=('field_accuracy', 'mean'),
                all_correct=('all_correct', 'mean'), seconds=('seconds_total', 'mean'))
           .round(3).reset_index())
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, dependency in zip(axes, ['independent', 'chained']):
    part = runs[runs.dependency == dependency]
    for (model, language), group in part.groupby(['model', 'language']):
        line = group.groupby('granularity').field_accuracy.mean()
        ax.plot([str(g) for g in line.index], line.values, marker='o', label=f'{model}, {language}')
    ax.set_title(f'{dependency} fields')
    ax.set_xlabel('Sub-calls per form')
    ax.set_ylim(0, 1.04)
axes[0].set_ylabel('Fields correct (share)')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()